In [1]:
from datasets import load_dataset
import numpy as np

# There is only one split on the hub
dataset = load_dataset("OGB/ogbg-molhiv", cache_dir="./data")
seed = 42
dataset = dataset.shuffle(seed=seed)

/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# activate notebook autoload of files
%load_ext autoreload
%autoreload 2

In [3]:
from transformers.models.graphormer.collating_graphormer import preprocess_item, GraphormerDataCollator

# dataset_processed = dataset.map(preprocess_item, batched=False)

In [4]:
from transformers import GraphormerForGraphClassification

model = GraphormerForGraphClassification.from_pretrained(
    "clefourrier/pcqm4mv2_graphormer_base",
    num_classes=2, # num_classes for the downstream task 
    ignore_mismatched_sizes=True,
)

/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of GraphormerForGraphClassification were not initialized from the model checkpoint at clefourrier/pcqm4mv2_graphormer_base and are newly initialized because the shapes did not match:
- classifier.classifier.weight: found shape torch.Size([1, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
from peft import LoraConfig
lora_config = LoraConfig(
    r=4,
    target_modules=["q_proj", "k_proj"]
)
model.add_adapter(adapter_config=lora_config)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    "graph-classification/lora/molhiv",
    logging_dir="graph-classification",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    auto_find_batch_size=True, # batch size can be changed automatically to prevent OOMs
    gradient_accumulation_steps=4, # simulating batch size of 64
    dataloader_num_workers=8, #1, 
    num_train_epochs=4,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",  # Save model checkpoint every epoch
    save_total_limit=3,  # Keep only the last 3 checkpoints to save disk space
    load_best_model_at_end=False,  # Disabled for LoRA compatibility
    metric_for_best_model="roc_auc",  # Use accuracy to determine the best model
    push_to_hub=False,
    use_mps_device=True,
    dataloader_drop_last=True,
    remove_unused_columns=False,  # CRITICAL: Don't remove columns - GraphormerDataCollator needs them!
    seed = seed
)

/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/transformers/training_args.py:2046: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(


In [7]:
train = dataset['train'].with_format("numpy")
eval = dataset['validation'].with_format("numpy")

In [ ]:
from typing import Dict
import evaluate
from transformers import EvalPrediction
from trainer import Trainer

metric = evaluate.load("roc_auc")

def compute_metrics(eval_pred: EvalPrediction) -> Dict:
    logits, labels = eval_pred.predictions, eval_pred.label_ids.reshape(-1)
    predictions = np.argmax(logits, axis=-1)
    predictions = np.reshape(predictions, -1)
    result = metric.compute(prediction_scores=predictions, references=labels)
    return result

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=eval,
    data_collator=GraphormerDataCollator(on_the_fly_processing=True),
    compute_metrics=compute_metrics
)
train_results = trainer.train(resume_from_checkpoint=True)

  0%|          | 0/2056 [00:00<?, ?it/s]/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Could not estimate the number of tokens of the input, floating-point operations will not be computed
 25%|██▌       | 514/2056 [22:11<6:05:13, 14.21s/it]

{'loss': 0.3075, 'grad_norm': 0.11883872002363205, 'learning_rate': 3.7500000000000003e-05, 'epoch': 1.0}


                                                    
 25%|██▌       | 514/2056 [25:07<6:05:13, 14.21s/it]/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/transformers/integrations/peft.py:391: FutureWarning: The `active_adapter` method is deprecated and will be removed in a future version.
  warnings.warn(


{'eval_loss': 0.13601323553444347, 'eval_roc_auc': 0.5, 'eval_runtime': 176.1359, 'eval_samples_per_second': 23.351, 'eval_steps_per_second': 1.465, 'epoch': 1.0}


/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
 36%|███▌      | 733/2056 [32:59<1:53:52,  5.16s/it] Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0x36c067e30>
    label = <none> 
    device = <AGXG13XDevice: 0x14ffe8400>
        name = Apple M1 Pro 
    commandQueue = <AGXG13XFamilyCommandQueue: 0x16b652400>
        label = <none> 
        device = <AGXG13XDevice: 0x14ffe8400>
            name = Apple M1 Pro 
    retainedReferences = 1
 36%|███▌      | 742/2056 [33:27<1:01:25,  2.81s/it]Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it ma

{'loss': 0.1736, 'grad_norm': 0.1296449899673462, 'learning_rate': 2.5e-05, 'epoch': 2.0}


                                                     
 50%|█████     | 1028/2056 [48:04<3:42:02, 12.96s/it]/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/transformers/integrations/peft.py:391: FutureWarning: The `active_adapter` method is deprecated and will be removed in a future version.
  warnings.warn(


{'eval_loss': 0.11119098149616895, 'eval_roc_auc': 0.5, 'eval_runtime': 121.1381, 'eval_samples_per_second': 33.953, 'eval_steps_per_second': 2.13, 'epoch': 2.0}


/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
 53%|█████▎    | 1084/2056 [50:15<50:30,  3.12s/it]   

: 